# NB06 — Train SAC (Apple Full-Body) — SOTA Edition

**Env:** `UnitreeG1PlaceAppleInBowlFullBody-v1`  
**Control:** `pd_joint_delta_pos` (37 DOF)  
**Reward:** `normalized_dense`

SOTA upgrades (beyond Gemini):
- Explicit `reward_mode="normalized_dense"`
- **No drifting VecNormalize** (replay buffer mismatch)
- Optional **stationary observation normalization** (fixed mean/std; safe for off-policy)
- `SiLU` + bigger net
- Optional action safety: jerk limiter + EMA + action-scale warmup
- Robust eval utilities (Gym env + SB3 VecEnv-style)
- Best checkpoint by success_rate + GIF recorder + failure taxonomy

Last updated: 2026-03-03 05:53


## 0) Imports & paths

In [ ]:

import os, sys, json, time, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import gymnasium as gym
import torch

from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback
from stable_baselines3.common.utils import set_random_seed

try:
    import imageio.v2 as imageio
except Exception:
    imageio = None

import mani_skill.envs
from mani_skill.utils.wrappers.gymnasium import CPUGymWrapper

PROJECT_ROOT = Path.cwd()
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "NB06_SAC_SOTA"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("ARTIFACTS_DIR:", ARTIFACTS_DIR)


## 1) Register custom env

In [ ]:

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from apple_fullbody_env import UnitreeG1PlaceAppleInBowlFullBodyEnv  # noqa: F401
print("✅ Imported apple_fullbody_env.py (custom env registered)")


## 2) Configuration

In [ ]:

IS_GPU = torch.cuda.is_available()
DEVICE = "cuda" if IS_GPU else "cpu"

CFG = {
    "seed": 42,

    "env_id": "UnitreeG1PlaceAppleInBowlFullBody-v1",
    "obs_mode": "state",
    "control_mode": "pd_joint_delta_pos",
    "reward_mode": "normalized_dense",
    "n_envs": 1,

    "total_env_steps": 2_000_000 if IS_GPU else 100_000,

    # SAC
    "learning_rate": 3e-4,
    "lr_end": 1e-5,
    "buffer_size": 5_000_000 if IS_GPU else 200_000,
    "batch_size": 1024 if IS_GPU else 256,
    "tau": 0.005,
    "gamma": 0.99,
    "ent_coef": "auto_0.1",
    "target_entropy": "auto",
    "learning_starts": 10_000,
    "train_freq": 1,
    "gradient_steps": 1,

    # Policy
    "activation": "SiLU",
    "net_arch": [512, 512, 256],

    # Stationary obs norm (safe)
    "use_fixed_obs_norm": True,
    "obs_norm_collect_steps": 20_000 if IS_GPU else 2_000,
    "obs_clip": 10.0,

    # Action safety
    "use_action_filter": True,
    "ema_alpha": 0.20,
    "jerk_clip": 0.08,
    "use_action_scale_warmup": True,
    "warmup_steps": 50_000,
    "start_scale": 0.35,

    # Checkpoint & eval
    "checkpoint_freq": 200_000,
    "eval_freq": 100_000,
    "eval_episodes": 20,

    # GIF
    "record_gif": True,
    "gif_fps": 20,
    "gif_max_frames": 300,
}

SEED = CFG["seed"]
set_random_seed(SEED)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

(ARTIFACTS_DIR / "nb06_config.json").write_text(json.dumps(CFG, indent=2))
print("✅ Saved config:", ARTIFACTS_DIR / "nb06_config.json")
print("DEVICE:", DEVICE, "IS_GPU:", IS_GPU)


## 3) Action safety wrappers

In [ ]:

class ActionFilterWrapper(gym.Wrapper):
    def __init__(self, env, ema_alpha=0.2, jerk_clip=0.08):
        super().__init__(env)
        self.ema_alpha = float(ema_alpha)
        self.jerk_clip = float(jerk_clip)
        self._prev = None
        self._ema = None

    def reset(self, **kwargs):
        out = self.env.reset(**kwargs)
        self._prev = np.zeros(self.action_space.shape, dtype=np.float32)
        self._ema = None
        return out

    def step(self, action):
        a = np.asarray(action, dtype=np.float32)
        if self._ema is None:
            self._ema = a.copy()
        else:
            self._ema = (1.0 - self.ema_alpha) * self._ema + self.ema_alpha * a
        d = np.clip(self._ema - self._prev, -self.jerk_clip, self.jerk_clip)
        out = self._prev + d
        self._prev = out.copy()
        return self.env.step(out.astype(np.float32))

class ActionScaleWarmupWrapper(gym.Wrapper):
    def __init__(self, env, start_scale=0.35, warmup_steps=50_000):
        super().__init__(env)
        self.start_scale = float(start_scale)
        self.warmup_steps = int(warmup_steps)
        self.t = 0

    def reset(self, **kwargs):
        self.t = 0
        return self.env.reset(**kwargs)

    def _scale(self):
        if self.warmup_steps <= 0:
            return 1.0
        p = min(1.0, self.t / self.warmup_steps)
        return self.start_scale + p * (1.0 - self.start_scale)

    def step(self, action):
        s = self._scale()
        self.t += 1
        return self.env.step(np.asarray(action, dtype=np.float32) * s)


## 4) Env factory (explicit normalized_dense)

In [ ]:

def make_env(for_eval=False):
    env = gym.make(
        CFG["env_id"],
        num_envs=1,
        obs_mode=CFG["obs_mode"],
        reward_mode=CFG["reward_mode"],
        control_mode=CFG["control_mode"],
        render_mode="rgb_array",
    )
    env = CPUGymWrapper(env)

    if CFG["use_action_scale_warmup"] and (not for_eval):
        env = ActionScaleWarmupWrapper(env, CFG["start_scale"], CFG["warmup_steps"])
    if CFG["use_action_filter"]:
        env = ActionFilterWrapper(env, CFG["ema_alpha"], CFG["jerk_clip"])

    return env

env_raw = make_env(for_eval=False)
eval_env_raw = make_env(for_eval=True)

print("✅ env_raw created")
print("obs space:", env_raw.observation_space)
print("act space:", env_raw.action_space)


## 5) Stationary observation normalization (safe for SAC)

In [ ]:

def collect_obs_stats(env, n_steps=10_000, seed=0):
    obs, info = env.reset(seed=seed)
    obs = np.asarray(obs, dtype=np.float32)

    mean = np.zeros_like(obs, dtype=np.float64)
    m2 = np.zeros_like(obs, dtype=np.float64)
    count = 0

    for t in range(n_steps):
        a = env.action_space.sample()
        obs, rew, terminated, truncated, info = env.step(a)
        x = np.asarray(obs, dtype=np.float32)

        count += 1
        delta = x - mean
        mean += delta / count
        delta2 = x - mean
        m2 += delta * delta2

        if terminated or truncated:
            obs, info = env.reset(seed=seed + t + 1)

    var = m2 / max(1, (count - 1))
    std = np.sqrt(var + 1e-8)
    return mean.astype(np.float32), std.astype(np.float32), int(count)

class FixedObsNormWrapper(gym.ObservationWrapper):
    def __init__(self, env, mean, std, clip=10.0):
        super().__init__(env)
        self.mean = mean.astype(np.float32)
        self.std = std.astype(np.float32)
        self.clip = float(clip)

    def observation(self, observation):
        x = np.asarray(observation, dtype=np.float32)
        z = (x - self.mean) / (self.std + 1e-8)
        z = np.clip(z, -self.clip, self.clip).astype(np.float32)
        return z

if CFG["use_fixed_obs_norm"]:
    print("Collecting fixed obs stats ...")
    mean, std, n = collect_obs_stats(env_raw, n_steps=CFG["obs_norm_collect_steps"], seed=123)
    np.savez_compressed(ARTIFACTS_DIR / "obs_norm_stats.npz", mean=mean, std=std, n=n)
    print("✅ Saved obs_norm_stats.npz")

    env = FixedObsNormWrapper(env_raw, mean, std, clip=CFG["obs_clip"])
    eval_env = FixedObsNormWrapper(eval_env_raw, mean, std, clip=CFG["obs_clip"])
else:
    env = env_raw
    eval_env = eval_env_raw

print("✅ Training env ready")


## 6) Robust eval utilities

In [ ]:

def _to_np(x):
    if x is None:
        return None
    if torch.is_tensor(x):
        x = x.detach().cpu().numpy()
    if isinstance(x, (list, tuple)):
        x = np.array(x)
    if isinstance(x, np.ndarray) and x.ndim == 2 and x.shape[0] == 1:
        x = x[0]
    return x

def get_extra(obs, info):
    if isinstance(info, dict) and any(k in info for k in ["tcp_to_apple","apple_to_bowl","is_grasped"]):
        out = {}
        for k in ["tcp_to_apple","apple_to_bowl","is_grasped","success","fail","dist_apple_bowl"]:
            if k in info:
                out[k] = _to_np(info.get(k))
        return out
    if isinstance(obs, dict) and "extra" in obs and isinstance(obs["extra"], dict):
        ex = obs["extra"]
        return {k:_to_np(ex.get(k)) for k in ["tcp_to_apple","apple_to_bowl","is_grasped"] if k in ex}
    return {}

def evaluate_model(model, env, n_episodes=20, seed0=0, max_steps=1000, record_gif_path=None):
    rows, frames = [], []
    do_gif = (record_gif_path is not None) and (imageio is not None) and CFG["record_gif"]

    for ep in range(n_episodes):
        obs, info = env.reset(seed=seed0 + ep * 101)
        ep_ret, steps = 0.0, 0
        final_info = {}
        min_t2a, min_a2b = 1e9, 1e9
        ever_grasped = False

        if do_gif and ep == 0:
            try:
                fr = env.render()
                if fr is not None:
                    frames.append(fr)
            except Exception:
                pass

        for t in range(max_steps):
            action, _ = model.predict(obs, deterministic=True)
            obs, rew, terminated, truncated, info = env.step(action)
            ep_ret += float(rew)
            steps = t + 1

            ex = get_extra(obs, info)
            if "tcp_to_apple" in ex:
                min_t2a = min(min_t2a, float(np.linalg.norm(np.array(ex["tcp_to_apple"]).reshape(-1))))
            if "dist_apple_bowl" in info:
                min_a2b = min(min_a2b, float(_to_np(info["dist_apple_bowl"]).reshape(-1)[0]))
            if "is_grasped" in ex:
                ever_grasped = ever_grasped or (float(np.array(ex["is_grasped"]).reshape(-1)[0]) > 0.5)

            if do_gif and ep == 0 and len(frames) < CFG["gif_max_frames"]:
                try:
                    fr = env.render()
                    if fr is not None:
                        frames.append(fr)
                except Exception:
                    pass

            if terminated or truncated:
                final_info = dict(info)
                break

        success = bool(final_info.get("success", False))
        fail_nan = bool(final_info.get("fail", False))
        timeout = bool(truncated) and (not terminated)

        rows.append({
            "episode": ep,
            "return": ep_ret,
            "steps": steps,
            "success": success,
            "fail_nan": fail_nan,
            "timeout": timeout,
            "min_dist_tcp_apple": min_t2a if min_t2a < 1e8 else np.nan,
            "min_dist_apple_bowl": min_a2b if min_a2b < 1e8 else np.nan,
            "ever_grasped": ever_grasped,
        })

    df = pd.DataFrame(rows)

    if do_gif and len(frames) > 0:
        imageio.mimsave(record_gif_path, frames, fps=CFG["gif_fps"])
        print("🎞️ Saved GIF:", record_gif_path)

    summary = {
        "success_rate": float(df["success"].mean()),
        "fail_nan_rate": float(df["fail_nan"].mean()),
        "timeout_rate": float(df["timeout"].mean()),
        "mean_return": float(df["return"].mean()),
        "mean_steps": float(df["steps"].mean()),
        "ever_grasped_rate": float(df["ever_grasped"].mean()),
        "mean_min_dist_tcp_apple": float(df["min_dist_tcp_apple"].mean(skipna=True)),
        "mean_min_dist_apple_bowl": float(df["min_dist_apple_bowl"].mean(skipna=True)),
    }
    return df, summary


## 7) Callbacks

In [ ]:

class AppleEvalCallback(BaseCallback):
    def __init__(self, eval_env, eval_freq, n_eval_episodes, log_dir: Path, verbose=1):
        super().__init__(verbose)
        self.eval_env = eval_env
        self.eval_freq = int(eval_freq)
        self.n_eval_episodes = int(n_eval_episodes)
        self.log_dir = Path(log_dir)
        self.best_success = -1.0
        self.history = []
        self._next_eval = self.eval_freq

    def _on_step(self) -> bool:
        if self.num_timesteps < self._next_eval:
            return True
        self._next_eval += self.eval_freq

        gif_path = self.log_dir / f"eval_sac_step_{self.num_timesteps}.gif"
        df, summary = evaluate_model(
            self.model, self.eval_env,
            n_episodes=self.n_eval_episodes,
            seed0=1234,
            record_gif_path=gif_path if CFG["record_gif"] else None
        )
        df.to_csv(self.log_dir / f"eval_sac_step_{self.num_timesteps}.csv", index=False)

        for k, v in summary.items():
            self.logger.record(f"eval/{k}", v)

        self.history.append({"timesteps": self.num_timesteps, **summary})

        if summary["success_rate"] > self.best_success:
            self.best_success = summary["success_rate"]
            self.model.save(str(self.log_dir / "best_sac_model"))
            if self.verbose:
                print(f"🏆 New best success_rate={self.best_success:.3f} @ {self.num_timesteps} timesteps")

        return True

class FailNaNMonitorCallback(BaseCallback):
    def __init__(self, verbose=0):
        super().__init__(verbose)
        self.count = 0

    def _on_step(self) -> bool:
        infos = self.locals.get("infos", [])
        for inf in infos:
            if isinstance(inf, dict) and inf.get("fail", False):
                self.count += 1
        self.logger.record("train/fail_nan_count", float(self.count))
        return True

checkpoint_cb = CheckpointCallback(
    save_freq=CFG["checkpoint_freq"],
    save_path=str(ARTIFACTS_DIR),
    name_prefix="sac_ckpt"
)
eval_cb = AppleEvalCallback(eval_env, CFG["eval_freq"], CFG["eval_episodes"], ARTIFACTS_DIR, verbose=1)
fail_cb = FailNaNMonitorCallback()

print("✅ Callbacks ready")


## 8) Build SAC model

In [ ]:

def linear_schedule(initial_lr: float, final_lr: float):
    def schedule(progress_remaining: float) -> float:
        return final_lr + (initial_lr - final_lr) * progress_remaining
    return schedule

lr_schedule = linear_schedule(CFG["learning_rate"], CFG["lr_end"])

policy_kwargs = dict(
    net_arch=CFG["net_arch"],
    activation_fn=torch.nn.SiLU,
)

model = SAC(
    "MlpPolicy",
    env,
    learning_rate=lr_schedule,
    buffer_size=CFG["buffer_size"],
    batch_size=CFG["batch_size"],
    tau=CFG["tau"],
    gamma=CFG["gamma"],
    ent_coef=CFG["ent_coef"],
    target_entropy=CFG["target_entropy"],
    learning_starts=CFG["learning_starts"],
    train_freq=CFG["train_freq"],
    gradient_steps=CFG["gradient_steps"],
    policy_kwargs=policy_kwargs,
    verbose=1,
    seed=SEED,
    device=DEVICE,
    tensorboard_log=str(ARTIFACTS_DIR / "tb"),
)

print("✅ SAC model created")


## 9) Train

In [ ]:

t0 = time.time()
model.learn(
    total_timesteps=CFG["total_env_steps"],
    callback=[checkpoint_cb, eval_cb, fail_cb],
    progress_bar=True
)
print(f"✅ Training done in {(time.time()-t0)/60:.1f} min")


## 10) Save final artifacts

In [ ]:

model.save(str(ARTIFACTS_DIR / "sac_final_model"))
print("✅ Saved:", ARTIFACTS_DIR / "sac_final_model.zip")

hist = pd.DataFrame(eval_cb.history)
hist.to_csv(ARTIFACTS_DIR / "eval_history.csv", index=False)
print("✅ Saved eval_history.csv")


## 11) Plot curves

In [ ]:

hist_path = ARTIFACTS_DIR / "eval_history.csv"
if hist_path.exists():
    hist = pd.read_csv(hist_path)
    if len(hist) > 0:
        fig, ax = plt.subplots(figsize=(9,4))
        ax.plot(hist["timesteps"], hist["success_rate"], marker="o")
        ax.set_title("SAC success_rate over training")
        ax.set_xlabel("timesteps")
        ax.set_ylabel("success_rate")
        fig.tight_layout()
        fig.savefig(ARTIFACTS_DIR / "plot_success_rate.png", dpi=160)
        plt.show()

        fig, ax = plt.subplots(figsize=(9,4))
        ax.plot(hist["timesteps"], hist["mean_return"], marker="o")
        ax.set_title("SAC mean_return over training")
        ax.set_xlabel("timesteps")
        ax.set_ylabel("mean_return (normalized_dense)")
        fig.tight_layout()
        fig.savefig(ARTIFACTS_DIR / "plot_mean_return.png", dpi=160)
        plt.show()


## 12) Final eval + error analysis

In [ ]:

final_gif = ARTIFACTS_DIR / "sac_final_eval.gif"
df_eval, summary = evaluate_model(model, eval_env, n_episodes=CFG["eval_episodes"], seed0=555, record_gif_path=final_gif)
df_eval.to_csv(ARTIFACTS_DIR / "sac_final_eval_episodes.csv", index=False)
(ARTIFACTS_DIR / "sac_final_eval_summary.json").write_text(json.dumps(summary, indent=2))

print("Final eval summary:")
for k, v in summary.items():
    print(f"  {k}: {v}")

def bucket(row):
    if row["fail_nan"]:
        return "fail_nan"
    if row["success"]:
        return "success"
    if row["timeout"]:
        return "timeout"
    dA = row["min_dist_tcp_apple"]
    grasp = row["ever_grasped"]
    dB = row["min_dist_apple_bowl"]
    if np.isnan(dA) or dA > 0.12:
        return "no_reach"
    if not grasp:
        return "reach_no_grasp"
    if np.isnan(dB) or dB > 0.15:
        return "grasp_no_place"
    return "other_fail"

df_eval["fail_type"] = df_eval.apply(bucket, axis=1)
fail_summary = df_eval["fail_type"].value_counts().rename_axis("fail_type").reset_index(name="count")
fail_summary.to_csv(ARTIFACTS_DIR / "sac_fail_summary.csv", index=False)
print("\nFailure breakdown:")
print(fail_summary)
